# Çözücüler ⚙️

Bu egzersizde, farklı `çözücülerin` `LogisticRegression` modelleri üzerindeki etkilerini araştıracaksınız.

👇 Veri kümesini içe aktarmak için aşağıdaki kodu çalıştırın

In [1]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/solvers_dataset.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,sulphates,alcohol,quality rating
0,9.47,5.97,7.36,10.17,6.84,9.15,9.78,9.52,10.34,8.80,6
1,10.05,8.84,9.76,8.38,10.15,6.91,9.70,9.01,9.23,8.80,7
2,10.59,10.71,10.84,10.97,9.03,10.42,11.46,11.25,11.34,9.06,4
3,11.00,8.44,8.32,9.65,7.87,10.92,6.97,11.07,10.66,8.89,8
4,12.12,13.44,10.35,9.95,11.09,9.38,10.22,9.04,7.68,11.38,3


In [2]:
df.shape, df.columns

((100000, 11),
 Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
        'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
        'sulphates', 'alcohol', 'quality rating'],
       dtype='object'))

In [3]:
df["quality rating"].value_counts().sort_index()

quality rating
1     10090
2     10030
3      9838
4      9928
5     10124
6      9961
7      9954
8      9977
9      9955
10    10143
Name: count, dtype: int64

- Veri kümesi farklı şaraplardan oluşmaktadır 🍷
- Özellikler şarapların farklı niteliklerini tanımlar 
- Hedef 🎯 bir uzman tarafından verilen kalite değerlendirmesidir

Büyük resim: “solver” ne ve neden performansı etkiliyor?

Lojistik Regresyon aslında bir optimizasyon problemi: model, hatayı (log-loss) azaltacak ağırlıkları bulmak için iteratif şekilde “en iyi noktaya” yürür.

Solver = bu yürümeyi yapan algoritma.
Farklı solver’lar:

farklı hızda yakınsar (daha çabuk biter / daha çok iterasyon ister),

bazı veri tiplerinde daha stabil çalışır,

bazıları L1/L2/ElasticNet gibi cezalara daha uygundur,

bazıları büyük veri için daha avantajlıdır.

Bu ödevde senden iki şey isteniyor:

En iyi precision (kesinlik) veren solver hangisi?

En kısa sürede yakınsayan (fit süresi en kısa) solver hangisi?

Precision = “pozitif dediklerimin ne kadarı gerçekten pozitif?”
Özellikle yanlış alarmın maliyetli olduğu senaryolarda önemlidir.

## 1. Hedef mühendisliği

Bu bölümde, değerlendirmeleri ikili bir hedefe dönüştüreceksiniz.

👇 Her değerlendirme için kaç gözlem bulunmaktadır?

In [7]:
len(df)


100000

In [4]:
df["is_good"] = (df["quality rating"] >= 6).astype(int)
df["is_good"].value_counts()


is_good
0    50010
1    49990
Name: count, dtype: int64

In [5]:
X = df.drop(columns=["quality rating", "is_good"])
y = df["is_good"]

X.shape, y.shape


((100000, 10), (100000,))

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape


((80000, 10), (20000, 10))

❓ Hedefi ikili sınıflandırma görevine dönüştürerek `y` oluşturun, burada 6'nın altındaki kalite değerlendirmeleri kötü [0], 6 ve üzeri değerlendirmeler iyi [1] olacak

In [ ]:
# YOUR CODE HERE

❓ Yeni ikili hedefin sınıf dengesini kontrol edin

In [ ]:
# YOUR CODE HERE

❓ Özellikleri normalleştirerek `X`'inizi oluşturun. Bu farklı çözücülerin adil karşılaştırılmasına olanak sağlayacaktır.

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape


((80000, 10), (20000, 10))

In [9]:
import time
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

solvers = ["liblinear", "lbfgs", "newton-cg", "sag", "saga"]

results = []

for s in solvers:
    model = LogisticRegression(solver=s, max_iter=5000, random_state=42)

    start = time.perf_counter()
    model.fit(X_train_scaled, y_train)
    fit_time = time.perf_counter() - start

    y_pred = model.predict(X_test_scaled)
    prec = precision_score(y_test, y_pred)

    n_iter = model.n_iter_
    n_iter = int(max(n_iter)) if hasattr(n_iter, "__len__") else int(n_iter)

    results.append({
        "solver": s,
        "precision": prec,
        "fit_time_sec": fit_time,
        "n_iter": n_iter
    })

results_df = pd.DataFrame(results).sort_values(by="precision", ascending=False)
results_df
# YOUR CODE HERE

,solver,precision,fit_time_sec,n_iter
0,liblinear,0.875156,0.126031,6
2,newton-cg,0.875156,0.068284,5
3,sag,0.875156,0.743969,67
4,saga,0.875156,1.347130,105
1,lbfgs,0.875052,0.068396,8


## 2. LogisticRegression çözücüleri

❓ Lojistik Regresyon modelleri farklı **çözücüler** kullanılarak optimize edilebilir. Mevcut çözücülerin karşılaştırmasını yapın:
- Uyum süresi - hangi çözücü **en hızlı**?
- Kesinlik - kesinlik puanları **ne kadar farklı**?

Lojistik Regresyon için mevcut çözücüler: `['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']`
 
Bu 5 çözücü hakkında daha fazla bilgi için [bu Stack Overflow konusuna](https://stackoverflow.com/questions/38640109/logistic-regression-python-solvers-defintions) göz atın

In [ ]:
# YOUR CODE HERE

In [10]:
# YOUR ANSWER
fastest_solver = "newton-cg"

Tüm solver’lar benzer precision üretti (~0.875), yani model performansı solver seçiminden çok etkilenmedi. Ancak fit süresi farklıydı. En hızlı solver newton-cg oldu (≈0.068 sn). lbfgs çok yakın ikinciydi. sag ve saga daha fazla iterasyon aldığı için daha yavaştı.

<details>
    <summary>ℹ️ Yorumumuz için buraya tıklayın</summary>

Maliyet fonksiyonumuz 5 çözücünün de bulduğu global bir minimuma sahip olacak kadar "kolay" olduğundan, tüm çözücüler benzer kesinlik puanları üretmelidir. Derin Öğrenme'de olduğu gibi çok karmaşık maliyet fonksiyonları için, farklı çözücüler kayıp fonksiyonunun farklı değerlerinde durabilir.

**Şarap veri kümesi**
    
Mevcut veri kümesinde sklearn'in <a href="https://scikit-learn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html">permutation_importance</a> ile özellik önemini kontrol ederseniz, birçok özelliğin neredeyse 0 önemine sahip olduğunu göreceksiniz. Liblinear çözücü, bir defada sadece *bir* yön boyunca hareket eder ve diğerlerini L1 düzenlileştirme ile düzenler (yani, beta değerlerini 0'a ayarlar), bu da birçok özelliğin hedefi tahmin etmede o kadar da önemli olmadığı bir veri kümesi için iyi bir uyum sağlayabilir.

❗️En iyi çözücüyü arama maliyeti vardır. Varsayılanla (`lbfgs`) devam etmek genel olarak en çok zaman tasarrufu sağlayabilir, sklearn başlamak için hangi çözücüyü seçeceğiniz konusunda fikir vermek için bu tabloyu sunar: 

<img src="https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/solvers-chart.png" width=700>

</details>

###  🧪 Kodunuzu test edin

In [11]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'solvers',
    fastest_solver=fastest_solver
)
result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/tumay/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/tumay/code/S16D4-S-data-solvers/tests
plugins: anyio-4.8.0, dash-3.3.0, typeguard-4.4.2
collecting ... collected 1 item

test_solvers.py::TestSolvers::test_fastest_solver PASSED                 [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/solvers.pickle

git commit -m 'Completed solvers step'

git push origin master



## 3. Stokastik Gradyan İnişi

Lojistik Regresyon modelleri ayrıca Stokastik Gradyan İnişi ile de optimize edilebilir.

❓ **Stokastik Gradyan İnişi** ile optimize edilmiş bir Lojistik Regresyon modelini değerlendirin. Kesinlik puanı ve eğitim süresi 2. bölümde eğitilen modellerin performansı ile nasıl karşılaştırılır?

<details>
<summary>💡 İpucu</summary>

- Takılırsanız, [SGDClassifier belgelerine](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) bakın!

</details>

In [ ]:
# YOUR CODE HERE

☝️ SGD modeli, benzer performans için en kısa sürelerden birine sahip olmalıdır (hatta `liblinear`'dan bile daha kısa olabilir). Bu, Gradyan İnişinin her dönemini aynı anda 100k satırı belleğe yüklemek yerine tek bir satırda gerçekleştirmenin doğrudan bir etkisidir.

## 4. Tahminler

❓ En iyi modeli (kısa uyum süresi ve yüksek kesinlik ile dengelenen) kullanarak aşağıdaki şarabın ikili kalitesini (0 veya 1) tahmin edin. Şunları kaydedin:
- `predicted_class`
- `predicted_proba_of_class` (yani modeliniz 1 sınıfını tahmin ettiyse, 1'in sınıf olması gerektiğine inanma olasılığı nedir, 0 ile 1 arasında olmalıdır)

In [ ]:
new_wine = pd.read_csv('https://d32aokrjazspmn.cloudfront.net/materials/solvers_new_wine.csv')
new_wine

In [ ]:
# YOUR CODE HERE

# 🏁  Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'new_data_prediction',
    predicted_class=predicted_class,
    predicted_proba_of_class=predicted_proba_of_class
)
result.write()
print(result.check())